# A/B Reranker trên Colab T4 (free) — Zalo LTR 2021

So `AITeamVN/Vietnamese_Reranker` (self-host, ứng viên "vượt trần") với `qwen3-rerank` (production API) trên benchmark công khai Zalo LTR (MIT).

**Trước khi chạy:** Runtime → Change runtime type → **T4 GPU**.

Chạy lần lượt từng cell. Cần: 1 GitHub PAT (repo private) + QWEN_API_KEY (chỉ cho arm production).

In [ ]:
# 1) Xác nhận có GPU T4. Nếu báo 'not connected to GPU' → Runtime > Change runtime type > T4 GPU
!nvidia-smi

In [ ]:
# 2) Clone repo private (dev) — dán GitHub PAT (Settings > Developer settings > Fine-grained token, quyền Contents: Read)
import os
GITHUB_TOKEN = ""  # <-- DÁN PAT vào đây
!git clone https://{GITHUB_TOKEN}@github.com/trungnguyen1618033/legal-guard-PH.git
os.chdir('/content/legal-guard-PH')
!git checkout dev && git log --oneline -1
print('CWD:', os.getcwd())

In [ ]:
# 3) Cài deps TỐI THIỂU vào python hệ thống của Colab (KHÔNG dùng uv → giữ torch-CUDA sẵn có).
#    sentence-transformers dùng luôn torch+CUDA của Colab. Còn lại chỉ để import qwen adapter.
!pip install -q sentence-transformers pydantic pydantic-settings httpx huggingface_hub

In [ ]:
# 4) SMOKE test 40 query trước (nhanh ~1 phút) — kiểm mọi thứ chạy được đã, rồi mới chạy full.
!python -m evaluation.rerank_ab --reranker hf:AITeamVN/Vietnamese_Reranker --limit 40

In [ ]:
# 5) ARM self-host FULL (788 query) — chạy trên GPU T4. Lưu report riêng để không bị arm sau đè.
!python -m evaluation.rerank_ab --reranker hf:AITeamVN/Vietnamese_Reranker
!cp evaluation/rerank_ab_report.json evaluation/rerank_ab_AITeamVN.json
print('Đã lưu evaluation/rerank_ab_AITeamVN.json')

In [ ]:
# 6) ARM production qwen3-rerank (API, cần key) — head-to-head trên CÙNG 788 query.
import os
os.environ['QWEN_API_KEY'] = ''  # <-- DÁN QWEN key
!python -m evaluation.rerank_ab --reranker qwen3-api
!cp evaluation/rerank_ab_report.json evaluation/rerank_ab_qwen.json
print('Đã lưu evaluation/rerank_ab_qwen.json')

In [ ]:
# 7) SO KẾT QUẢ + verdict
import json
a = json.load(open('evaluation/rerank_ab_AITeamVN.json'))
q = json.load(open('evaluation/rerank_ab_qwen.json'))
base = a['bm25_baseline']['mrr@10']
am, qm = a['reranked']['mrr@10'], q['reranked']['mrr@10']
print(f"BM25 baseline        MRR@10 = {base:.4f}")
print(f"AITeamVN (self-host) MRR@10 = {am:.4f}  (lift {am-base:+.4f})  Recall@10={a['reranked']['recall@10']:.1%}")
print(f"qwen3-rerank (prod)  MRR@10 = {qm:.4f}  (lift {qm-base:+.4f})  Recall@10={q['reranked']['recall@10']:.1%}")
print()
diff = am - qm
if diff > 0.02:
    print(f"=> AITeamVN THẮNG rõ (+{diff:.4f}). Cân nhắc self-host (TEI) NẾU tải rerank cao.")
elif diff < -0.02:
    print(f"=> qwen3-rerank vẫn hơn ({diff:.4f}). Giữ API. Đóng hồ sơ A/B.")
else:
    print(f"=> Ngang nhau ({diff:+.4f}). Giữ API (rẻ 10-100x, zero-ops). Đóng hồ sơ A/B.")

In [ ]:
# 8) Tải 2 report về máy để lưu vào repo (git)
from google.colab import files
files.download('evaluation/rerank_ab_AITeamVN.json')
files.download('evaluation/rerank_ab_qwen.json')

---
# PHẦN 2 — Đo ACCURACY END-TO-END (câu hỏi thật: AITeamVN có nâng 98.1% không?)

Benchmark ở trên (MRR) đo *ranking retrieval*, KHÁC accuracy sản phẩm trên 54 golden. Phần này serve
AITeamVN dạng endpoint `/rerank` (TEI-compatible) + mở tunnel công khai → **máy local** chạy
`accuracy_eval` trỏ vào URL đó, so với 53/54.

Chạy cell 9→11 trên Colab (giữ notebook MỞ để tunnel sống), rồi làm theo hướng dẫn cell 12 ở máy local.

In [ ]:
# 9) Cài server nhẹ + cloudflared (tunnel công khai, KHÔNG cần signup)
!pip install -q flask
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
print('OK')

In [ ]:
# 10) Serve AITeamVN dưới dạng /rerank — IDEMPOTENT (chạy lại được, tự giải phóng port + tự xác nhận)
import subprocess
import threading
import time

import requests
from flask import Flask, jsonify, request
from sentence_transformers import CrossEncoder

# Giải phóng port 8080 nếu lần chạy trước còn giữ (tránh "Address already in use" → server cũ hỏng giữ port).
subprocess.run("fuser -k 8080/tcp", shell=True, stderr=subprocess.DEVNULL)
time.sleep(1)

ce = CrossEncoder('AITeamVN/Vietnamese_Reranker', max_length=512)   # ~500MB, tải vài phút lần đầu
srv = Flask(__name__)


@srv.route('/rerank', methods=['POST'])
def _rerank():
    b = request.get_json(force=True)
    q, texts = b['query'], b['texts']
    scores = ce.predict([[q, t] for t in texts])
    return jsonify([{'index': i, 'score': float(s)} for i, s in enumerate(scores)])


threading.Thread(target=lambda: srv.run(port=8080, threaded=True, use_reloader=False), daemon=True).start()

# Tự xác nhận route /rerank THẬT SỰ phục vụ (không chỉ "đã start") trước khi sang cell 11.
ready = False
for _ in range(15):
    time.sleep(2)
    try:
        r = requests.post("http://127.0.0.1:8080/rerank",
                          json={"query": "phạt", "texts": ["Điều 301 phạt 8%", "abc"]}, timeout=10)
        if r.status_code == 200 and isinstance(r.json(), list):
            ready = True
            break
    except Exception:
        pass
print("✅ server /rerank SẴN SÀNG ở :8080 — sang cell 11" if ready
      else "❌ server CHƯA sẵn sàng — Restart runtime rồi chạy lại cell này (đừng chạy 2 lần)")

In [ ]:
# 11) Mở tunnel công khai → COPY URL https in ra, dùng làm RERANK_URL ở máy local
import re
import subprocess
import threading
import time

import requests

proc = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:8080', '--no-autoupdate'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in proc.stdout:
    m = re.search(r'https://[-\w.]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break
threading.Thread(target=lambda: [None for _ in proc.stdout], daemon=True).start()  # drain, tránh nghẽn buffer

# Tunnel vừa in URL nhưng DNS/edge cloudflare cần vài giây mới resolve → chờ + retry, KHÔNG để self-test chặn.
ok = False
for i in range(10):
    time.sleep(5)
    try:
        r = requests.post(url + '/rerank',
                          json={'query': 'phạt vi phạm hợp đồng tối đa bao nhiêu',
                                'texts': ['Điều 301 phạt vi phạm tối đa 8%', 'điều khoản không liên quan']},
                          timeout=15)
        print(f'self-test /rerank OK (sau {(i+1)*5}s):', r.json())
        ok = True
        break
    except Exception as e:
        print(f'  chờ tunnel sẵn sàng… ({(i+1)*5}s) {type(e).__name__}')
if not ok:
    print('⚠️ self-test chưa qua (tunnel có thể cần thêm thời gian) — vẫn dùng URL dưới, thử lại lệnh local sau ~30s.')
print('\n>>> RERANK_URL =', url, '  (GIỮ notebook MỞ để tunnel sống)')

## 12) Ở MÁY LOCAL — chạy accuracy_eval, so apples-to-apples

Giữ notebook Colab MỞ (tunnel phải sống). Ở terminal repo local (config KHỚP /trust: sqlite + closure + rerank):

```bash
# A) BASELINE hiện tại (qwen3-rerank) — tái tạo ~53–54/54
DATABASE_URL="sqlite:///data/cases.db" CITATION_CLOSURE=1 CROSS_ENCODER_RERANK=true RERANK_URL= PERSIST_EMBEDDINGS=1 \
  uv run python -m evaluation.accuracy_eval --no-write --repeat=3

# B) CANDIDATE (AITeamVN self-host qua Colab) — DÁN url từ cell 11 vào RERANK_URL
DATABASE_URL="sqlite:///data/cases.db" CITATION_CLOSURE=1 CROSS_ENCODER_RERANK=true \
  RERANK_URL=https://XXXX.trycloudflare.com PERSIST_EMBEDDINGS=1 \
  uv run python -m evaluation.accuracy_eval --no-write --repeat=3
```

So `passed/54` của B với A (RERANK_URL set → dùng HttpReranker=AITeamVN; rỗng → qwen3-rerank API):
- **B > A rõ** → AITeamVN nâng accuracy thật → đáng deploy (TEI trên Alibaba GPU) khi mở rộng KB.
- **B ≈ A** → benchmark thắng KHÔNG dịch chuyển sản phẩm (KB nhỏ sát trần) → giữ qwen3-rerank API, để dành AITeamVN cho lúc corpus lớn.

`--no-write` để KHÔNG đè `accuracy_report.json`/`/trust` (đây là thí nghiệm). Mỗi arm ~162 call, vài phút.
